# LLM Usage & Cost Analytics

This notebook profiles request volume, token consumption, estimated cost, latency, and reliability. Costs use the illustrative rates documented in `README.md`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
from llm_analytics import load_usage_data, summarize, by_dimension
df = load_usage_data(ROOT / 'data' / 'llm_usage.csv')
df.head()

In [ ]:
summary = summarize(df)
pd.Series(summary)

## Model and application mix

The model table makes the cost drivers visible: premium models can dominate spend even when they handle fewer requests. Application-level aggregation highlights where optimization work will have the largest operational impact.

In [ ]:
model_summary = by_dimension(df, 'model')
application_summary = by_dimension(df, 'application')
display(model_summary)
display(application_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=model_summary, x='cost_usd', y='model', ax=axes[0], color='#4472c4')
axes[0].set_title('Estimated cost by model'); axes[0].set_xlabel('USD')
daily = df.groupby('date', as_index=False).agg(cost_usd=('cost_usd', 'sum'), requests=('request_id', 'count'))
sns.lineplot(data=daily, x='date', y='cost_usd', marker='o', ax=axes[1])
axes[1].set_title('Daily estimated cost'); axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()

## Reliability and latency findings

Compare success rate with p95 latency instead of relying on averages alone. Timeout requests create a clear long-latency tail; production applications should be prioritized for timeout mitigation. Token volume and cost should be monitored together because output tokens are priced differently across models.

In [ ]:
reliability = df.groupby('application').agg(
    requests=('request_id', 'count'),
    success_rate=('status', lambda s: (s == 'success').mean() * 100),
    p95_latency_ms=('latency_ms', lambda s: s.quantile(.95)),
    total_tokens=('total_tokens', 'sum'),
    cost_usd=('cost_usd', 'sum'),
).sort_values('success_rate')
reliability